# RiskOptima Dislocation Model

This notebook demonstrates `riskoptima.dislocation`. Equities use quality, valuation and a simplified FCF-proxy scenario; funds use historical price/regime dislocation without corporate DCF or a claimed NAV discount. Scores are research heuristics, not calibrated investment forecasts.


### Execution environment

Run from the repository root with the checkout installed (`python -m pip install -e .`). By default this notebook fetches Yahoo Finance data. Set `RISKOPTIMA_DEMO_LIVE=0` before execution for explicitly synthetic, deterministic inputs. This is never a silent fallback for failed live data. Individual ticker failures appear as ERROR rows; an entirely unsuccessful scan fails the notebook.


In [3]:
from IPython.display import Markdown, display
import pandas as pd
import os
from examples.example_dislocation_model import SyntheticProvider

from riskoptima.dislocation import (
    DislocationConfig,
    analyze_symbol,
    generate_markdown_report,
    scan_universe,
)

pd.set_option("display.max_columns", 30)
USE_LIVE_DATA = os.environ.get("RISKOPTIMA_DEMO_LIVE", "1") != "0"
provider = None if USE_LIVE_DATA else SyntheticProvider()
print("Data source:", "Yahoo Finance (live)" if USE_LIVE_DATA else "Synthetic, not market data")


Data source: Yahoo Finance (live)


## Input universe

The symbols below are Yahoo identifiers, not ISINs. Yahoo's ISIN search resolves IE00B50W2R13 to `0P0000KM24.L` (Global Bond Index GBP-hedged accumulation share class). Verify share class and currency before replacing an identifier.


In [5]:
INSTRUMENTS = [
    ("0P0000KM24.L", "Global Bond Index Fund GBP Hedged Acc"),
    ("0P0000TKZI.L", "LifeStrategy 40% Equity Fund"),
    ("0P0000TKZO.L", "LifeStrategy 100% Equity Fund"),
    ("0P0000TKZM.L", "LifeStrategy 80% Equity Fund"),
    ("0P00017UWT.L", "Target Retirement 2045 Fund"),
    ("0P0000UGLG.L", "ESG Developed World All Cap Equity Index Fund"),
    ("0P0000UGLE.L", "ESG Developed Europe Index Fund"),
    ("0P0000KM22.L", "Emerging Markets Stock Index Fund"),
    ("0P0000KM20.L", "Pacific ex-Japan Stock Index Fund"),
    ("0P00018XAP.L", "FTSE 100 Index Unit Trust"),
    ("0P0000KSPA.L", "U.S. Equity Index Fund"),
]

pd.DataFrame(INSTRUMENTS, columns=["symbol", "name"])


,symbol,name
0,0P0000KM24.L,Global Bond Index Fund GBP Hedged Acc
1,0P0000TKZI.L,LifeStrategy 40% Equity Fund
2,0P0000TKZO.L,LifeStrategy 100% Equity Fund
3,0P0000TKZM.L,LifeStrategy 80% Equity Fund
4,0P00017UWT.L,Target Retirement 2045 Fund
5,0P0000UGLG.L,ESG Developed World All Cap Equity Index Fund
6,0P0000UGLE.L,ESG Developed Europe Index Fund
7,0P0000KM22.L,Emerging Markets Stock Index Fund
8,0P0000KM20.L,Pacific ex-Japan Stock Index Fund
9,0P00018XAP.L,FTSE 100 Index Unit Trust


## Run and rank all instruments

The engine combines price drawdown, trend deviation, volatility and regime diagnostics. Data coverage measures input completeness, not model accuracy. Markov fitting uses a fixed seed; a fit failure is recorded in diagnostics before the momentum/volatility fallback is used.


In [7]:
config = DislocationConfig(history_period="5y", min_observations=126, use_markov_regime=True)
summary, reports = scan_universe(INSTRUMENTS, config=config, provider=provider, continue_on_error=True)
assert reports, "No successful analyses; inspect the ticker error rows"
print(f"Successful analyses: {len(reports)}/{len(INSTRUMENTS)}")
display(summary.loc[summary["classification"] == "ERROR", ["symbol", "error"]])
summary


Successful analyses: 11/11


,symbol,error


,symbol,name,instrument_type,price,currency,opportunity_score,dislocation_score,quality_score,valuation_score,confidence_score,classification,valuation_mode,error
0,0P0000UGLE.L,ESG Developed Europe Index Fund,MUTUAL_FUND,358.330414,GBP,38.893598,30.477191,None,None,100.0,EXTENDED,market_dislocation,None
1,0P0000KM24.L,Global Bond Index Fund GBP Hedged Acc,MUTUAL_FUND,156.373093,GBP,34.272830,36.532556,None,None,100.0,EXTENDED,market_dislocation,None
2,0P0000KM22.L,Emerging Markets Stock Index Fund,MUTUAL_FUND,416.557587,GBP,31.476099,18.028573,None,None,100.0,EXTENDED,market_dislocation,None
3,0P00018XAP.L,FTSE 100 Index Unit Trust,MUTUAL_FUND,223.542099,GBP,28.899049,13.980149,None,None,100.0,EXTENDED,market_dislocation,None
4,0P0000TKZI.L,LifeStrategy 40% Equity Fund,MUTUAL_FUND,231.223999,GBP,27.990016,15.388904,None,None,100.0,EXTENDED,market_dislocation,None
5,0P0000KM20.L,Pacific ex-Japan Stock Index Fund,MUTUAL_FUND,460.469086,GBP,27.467217,21.094796,None,None,100.0,EXTENDED,market_dislocation,None
6,0P00017UWT.L,Target Retirement 2045 Fund,MUTUAL_FUND,270.819794,GBP,20.662382,6.593836,None,None,100.0,EXTENDED,market_dislocation,None
7,0P0000TKZM.L,LifeStrategy 80% Equity Fund,MUTUAL_FUND,391.049805,GBP,19.764217,5.418714,None,None,100.0,EXTENDED,market_dislocation,None
8,0P0000TKZO.L,LifeStrategy 100% Equity Fund,MUTUAL_FUND,500.494598,GBP,19.352572,3.731522,None,None,100.0,EXTENDED,market_dislocation,None
9,0P0000UGLG.L,ESG Developed World All Cap Equity Index Fund,MUTUAL_FUND,604.013916,GBP,16.310081,3.601425,None,None,100.0,EXTENDED,market_dislocation,None


## Generate the full report for every successful ticker

The default prose is generated from the measured state and is reproducible. Fund reports disclose optional provider assets, yield, expense ratio, P/E and NAV reference fields, with explicit gaps and limitations. An equity fund has equity exposure, but its units are not corporate shares; these descriptors do not trigger corporate DCF or alter the score. Data coverage is not complete fund valuation coverage. Every report includes a research-only, not-investment-advice notice.

`generate_markdown_report` also accepts a `narrative_generator` callback, so an LLM can be injected by an application without adding an LLM SDK dependency to RiskOptima. The research notice remains present even with custom narratives.


In [9]:
generated_reports = {report.symbol: generate_markdown_report(report) for report in reports}

for symbol, report_markdown in generated_reports.items():
    display(Markdown(report_markdown))
    display(Markdown("---"))


# RiskOptima Dislocation Report — Global Bond Index Fund GBP Hedged Acc

**Ticker:** `0P0000KM24.L`  
**As of:** 2026-10-08  
**Instrument type:** MUTUAL_FUND  
**Valuation mode:** market_dislocation  
**Price:** 156.37 GBP  

## Executive signal

- **Opportunity score:** 34.3/100
- **Classification:** EXTENDED
- **Data coverage:** 100.0/100 (price history/basic identifiers, not complete fund valuation or a probability of accuracy)
- **Dislocation score:** 36.5/100
- **Regime opportunity:** 14.8/100

## Generated interpretation

Global Bond Index Fund GBP Hedged Acc (0P0000KM24.L) scores 34.3/100 and is classified as extended. This is a pooled fund, which may hold equities; its units are not shares in an operating company. RiskOptima does not fabricate a company-style intrinsic value for the fund. Available fund metadata is disclosed separately; the score uses price, drawdown, volatility and regimes. The market-dislocation component is 36.5/100 and the regime-opportunity component is 14.8/100. Data coverage is 100.0/100, not a calibrated probability of model accuracy.

## Market dislocation

| Metric | Value |
|---|---:|
| 1Y drawdown | -3.6% |
| Period drawdown | -4.9% |
| 200D trend deviation | -1.9% |
| Price z-score | -2.23 |
| 3M return | -2.2% |
| 12M return | -1.3% |
| Annualized volatility | 3.3% |
| Volatility percentile | 19.1% |
| Regime | strong |

## Valuation treatment

This instrument is not an operating-company equity. A fund may hold equities, but its units are not corporate shares. RiskOptima deliberately does **not** apply a corporate DCF to the fund itself. The opportunity score measures historical price and regime dislocation, not company intrinsic value. Fund metadata alone does not establish holdings-level fair value; no NAV premium or discount is measured.

## Fund metadata (provider-reported)

| Metric | Value |
|---|---:|
| Total assets (provider units; currency not independently verified) | n/a |
| Distribution yield | 2.4% |
| Annual-report expense ratio | n/a |
| Reported fund P/E (methodology unverified) | n/a |
| NAV reference (date/share class unverified) | n/a |
| Category | n/a |
| Fund family | n/a |

These optional fields are descriptive and do not change the dislocation score. Missing values are shown as n/a. Metadata may differ in date or units from the price history.

## Diagnostics

- Fund metadata coverage: 1/7 optional fields available; missing fields are not estimated.
- Data coverage score reflects price history and basic identifiers, not complete fund valuation data.
- No separate NAV supplied; no NAV premium or discount is measured.
- Fund units may provide equity exposure, but are not corporate shares. Corporate DCF is not applicable to the fund itself; holdings-level fair values and weights would be required for look-through valuation.

## Model note

The score is designed to detect potentially attractive dislocations, not to forecast an exact market bottom. Interpret scores alongside input coverage and, for corporate equities, fundamental quality; neither establishes predictive accuracy.

## Research notice

RiskOptima is for research and educational use only. Reports and scores are not investment advice or recommendations to buy, sell or hold any instrument. Models and third-party data may be incomplete or inaccurate. Past performance does not guarantee future results. Independently verify inputs and results before making investment decisions.

Generated by RiskOptima v2.8.1


---

# RiskOptima Dislocation Report — LifeStrategy 40% Equity Fund

**Ticker:** `0P0000TKZI.L`  
**As of:** 2026-10-08  
**Instrument type:** MUTUAL_FUND  
**Valuation mode:** market_dislocation  
**Price:** 231.22 GBP  

## Executive signal

- **Opportunity score:** 28.0/100
- **Classification:** EXTENDED
- **Data coverage:** 100.0/100 (price history/basic identifiers, not complete fund valuation or a probability of accuracy)
- **Dislocation score:** 15.4/100
- **Regime opportunity:** 50.3/100

## Generated interpretation

LifeStrategy 40% Equity Fund (0P0000TKZI.L) scores 28.0/100 and is classified as extended. This is a pooled fund, which may hold equities; its units are not shares in an operating company. RiskOptima does not fabricate a company-style intrinsic value for the fund. Available fund metadata is disclosed separately; the score uses price, drawdown, volatility and regimes. The market-dislocation component is 15.4/100 and the regime-opportunity component is 50.3/100. Data coverage is 100.0/100, not a calibrated probability of model accuracy.

## Market dislocation

| Metric | Value |
|---|---:|
| 1Y drawdown | -1.4% |
| Period drawdown | -1.4% |
| 200D trend deviation | 1.2% |
| Price z-score | 0.08 |
| 3M return | -0.4% |
| 12M return | 5.8% |
| Annualized volatility | 4.9% |
| Volatility percentile | 27.9% |
| Regime | neutral |

## Valuation treatment

This instrument is not an operating-company equity. A fund may hold equities, but its units are not corporate shares. RiskOptima deliberately does **not** apply a corporate DCF to the fund itself. The opportunity score measures historical price and regime dislocation, not company intrinsic value. Fund metadata alone does not establish holdings-level fair value; no NAV premium or discount is measured.

## Fund metadata (provider-reported)

| Metric | Value |
|---|---:|
| Total assets (provider units; currency not independently verified) | 5653216256.00 |
| Distribution yield | 0.0% |
| Annual-report expense ratio | n/a |
| Reported fund P/E (methodology unverified) | 19.70 |
| NAV reference (date/share class unverified) | n/a |
| Category | n/a |
| Fund family | n/a |

These optional fields are descriptive and do not change the dislocation score. Missing values are shown as n/a. Metadata may differ in date or units from the price history.

## Diagnostics

- Fund metadata coverage: 3/7 optional fields available; missing fields are not estimated.
- Data coverage score reflects price history and basic identifiers, not complete fund valuation data.
- Provider-reported fund P/E is descriptive; holdings methodology and valuation date are not verified.
- No separate NAV supplied; no NAV premium or discount is measured.
- Fund units may provide equity exposure, but are not corporate shares. Corporate DCF is not applicable to the fund itself; holdings-level fair values and weights would be required for look-through valuation.

## Model note

The score is designed to detect potentially attractive dislocations, not to forecast an exact market bottom. Interpret scores alongside input coverage and, for corporate equities, fundamental quality; neither establishes predictive accuracy.

## Research notice

RiskOptima is for research and educational use only. Reports and scores are not investment advice or recommendations to buy, sell or hold any instrument. Models and third-party data may be incomplete or inaccurate. Past performance does not guarantee future results. Independently verify inputs and results before making investment decisions.

Generated by RiskOptima v2.8.1


---

# RiskOptima Dislocation Report — LifeStrategy 100% Equity Fund

**Ticker:** `0P0000TKZO.L`  
**As of:** 2026-10-08  
**Instrument type:** MUTUAL_FUND  
**Valuation mode:** market_dislocation  
**Price:** 500.49 GBP  

## Executive signal

- **Opportunity score:** 19.4/100
- **Classification:** EXTENDED
- **Data coverage:** 100.0/100 (price history/basic identifiers, not complete fund valuation or a probability of accuracy)
- **Dislocation score:** 3.7/100
- **Regime opportunity:** 36.8/100

## Generated interpretation

LifeStrategy 100% Equity Fund (0P0000TKZO.L) scores 19.4/100 and is classified as extended. This is a pooled fund, which may hold equities; its units are not shares in an operating company. RiskOptima does not fabricate a company-style intrinsic value for the fund. Available fund metadata is disclosed separately; the score uses price, drawdown, volatility and regimes. The market-dislocation component is 3.7/100 and the regime-opportunity component is 36.8/100. Data coverage is 100.0/100, not a calibrated probability of model accuracy.

## Market dislocation

| Metric | Value |
|---|---:|
| 1Y drawdown | -0.9% |
| Period drawdown | -0.9% |
| 200D trend deviation | 6.3% |
| Price z-score | 1.14 |
| 3M return | 2.4% |
| 12M return | 16.9% |
| Annualized volatility | 8.7% |
| Volatility percentile | 23.7% |
| Regime | neutral |

## Valuation treatment

This instrument is not an operating-company equity. A fund may hold equities, but its units are not corporate shares. RiskOptima deliberately does **not** apply a corporate DCF to the fund itself. The opportunity score measures historical price and regime dislocation, not company intrinsic value. Fund metadata alone does not establish holdings-level fair value; no NAV premium or discount is measured.

## Fund metadata (provider-reported)

| Metric | Value |
|---|---:|
| Total assets (provider units; currency not independently verified) | 13042328576.00 |
| Distribution yield | 0.0% |
| Annual-report expense ratio | n/a |
| Reported fund P/E (methodology unverified) | 20.12 |
| NAV reference (date/share class unverified) | n/a |
| Category | n/a |
| Fund family | n/a |

These optional fields are descriptive and do not change the dislocation score. Missing values are shown as n/a. Metadata may differ in date or units from the price history.

## Diagnostics

- Fund metadata coverage: 3/7 optional fields available; missing fields are not estimated.
- Data coverage score reflects price history and basic identifiers, not complete fund valuation data.
- Provider-reported fund P/E is descriptive; holdings methodology and valuation date are not verified.
- No separate NAV supplied; no NAV premium or discount is measured.
- Fund units may provide equity exposure, but are not corporate shares. Corporate DCF is not applicable to the fund itself; holdings-level fair values and weights would be required for look-through valuation.

## Model note

The score is designed to detect potentially attractive dislocations, not to forecast an exact market bottom. Interpret scores alongside input coverage and, for corporate equities, fundamental quality; neither establishes predictive accuracy.

## Research notice

RiskOptima is for research and educational use only. Reports and scores are not investment advice or recommendations to buy, sell or hold any instrument. Models and third-party data may be incomplete or inaccurate. Past performance does not guarantee future results. Independently verify inputs and results before making investment decisions.

Generated by RiskOptima v2.8.1


...

---

# RiskOptima Dislocation Report — FTSE 100 Index Unit Trust

**Ticker:** `0P00018XAP.L`  
**As of:** 2026-10-09  
**Instrument type:** MUTUAL_FUND  
**Valuation mode:** market_dislocation  
**Price:** 223.54 GBP  

## Executive signal

- **Opportunity score:** 28.9/100
- **Classification:** EXTENDED
- **Data coverage:** 100.0/100 (price history/basic identifiers, not complete fund valuation or a probability of accuracy)
- **Dislocation score:** 14.0/100
- **Regime opportunity:** 49.6/100

## Generated interpretation

FTSE 100 Index Unit Trust (0P00018XAP.L) scores 28.9/100 and is classified as extended. This is a pooled fund, which may hold equities; its units are not shares in an operating company. RiskOptima does not fabricate a company-style intrinsic value for the fund. Available fund metadata is disclosed separately; the score uses price, drawdown, volatility and regimes. The market-dislocation component is 14.0/100 and the regime-opportunity component is 49.6/100. Data coverage is 100.0/100, not a calibrated probability of model accuracy.

## Market dislocation

| Metric | Value |
|---|---:|
| 1Y drawdown | -2.9% |
| Period drawdown | -2.9% |
| 200D trend deviation | 2.1% |
| Price z-score | 0.28 |
| 3M return | 1.0% |
| 12M return | 16.0% |
| Annualized volatility | 10.1% |
| Volatility percentile | 18.7% |
| Regime | neutral |

## Valuation treatment

This instrument is not an operating-company equity. A fund may hold equities, but its units are not corporate shares. RiskOptima deliberately does **not** apply a corporate DCF to the fund itself. The opportunity score measures historical price and regime dislocation, not company intrinsic value. Fund metadata alone does not establish holdings-level fair value; no NAV premium or discount is measured.

## Fund metadata (provider-reported)

| Metric | Value |
|---|---:|
| Total assets (provider units; currency not independently verified) | 2187394560.00 |
| Distribution yield | 2.8% |
| Annual-report expense ratio | n/a |
| Reported fund P/E (methodology unverified) | 16.58 |
| NAV reference (date/share class unverified) | n/a |
| Category | n/a |
| Fund family | n/a |

These optional fields are descriptive and do not change the dislocation score. Missing values are shown as n/a. Metadata may differ in date or units from the price history.

## Diagnostics

- Fund metadata coverage: 3/7 optional fields available; missing fields are not estimated.
- Data coverage score reflects price history and basic identifiers, not complete fund valuation data.
- Provider-reported fund P/E is descriptive; holdings methodology and valuation date are not verified.
- No separate NAV supplied; no NAV premium or discount is measured.
- Fund units may provide equity exposure, but are not corporate shares. Corporate DCF is not applicable to the fund itself; holdings-level fair values and weights would be required for look-through valuation.

## Model note

The score is designed to detect potentially attractive dislocations, not to forecast an exact market bottom. Interpret scores alongside input coverage and, for corporate equities, fundamental quality; neither establishes predictive accuracy.

## Research notice

RiskOptima is for research and educational use only. Reports and scores are not investment advice or recommendations to buy, sell or hold any instrument. Models and third-party data may be incomplete or inaccurate. Past performance does not guarantee future results. Independently verify inputs and results before making investment decisions.

Generated by RiskOptima v2.8.1


---

# RiskOptima Dislocation Report — U.S. Equity Index Fund

**Ticker:** `0P0000KSPA.L`  
**As of:** 2026-10-08  
**Instrument type:** MUTUAL_FUND  
**Valuation mode:** market_dislocation  
**Price:** 1318.01 GBP  

## Executive signal

- **Opportunity score:** 11.4/100
- **Classification:** EXTENDED
- **Data coverage:** 100.0/100 (price history/basic identifiers, not complete fund valuation or a probability of accuracy)
- **Dislocation score:** 1.3/100
- **Regime opportunity:** 5.7/100

## Generated interpretation

U.S. Equity Index Fund (0P0000KSPA.L) scores 11.4/100 and is classified as extended. This is a pooled fund, which may hold equities; its units are not shares in an operating company. RiskOptima does not fabricate a company-style intrinsic value for the fund. Available fund metadata is disclosed separately; the score uses price, drawdown, volatility and regimes. The market-dislocation component is 1.3/100 and the regime-opportunity component is 5.7/100. Data coverage is 100.0/100, not a calibrated probability of model accuracy.

## Market dislocation

| Metric | Value |
|---|---:|
| 1Y drawdown | -0.5% |
| Period drawdown | -0.5% |
| 200D trend deviation | 8.7% |
| Price z-score | 1.54 |
| 3M return | 3.5% |
| 12M return | 16.7% |
| Annualized volatility | 11.2% |
| Volatility percentile | 8.9% |
| Regime | strong |

## Valuation treatment

This instrument is not an operating-company equity. A fund may hold equities, but its units are not corporate shares. RiskOptima deliberately does **not** apply a corporate DCF to the fund itself. The opportunity score measures historical price and regime dislocation, not company intrinsic value. Fund metadata alone does not establish holdings-level fair value; no NAV premium or discount is measured.

## Fund metadata (provider-reported)

| Metric | Value |
|---|---:|
| Total assets (provider units; currency not independently verified) | 21030432768.00 |
| Distribution yield | 0.8% |
| Annual-report expense ratio | n/a |
| Reported fund P/E (methodology unverified) | 24.93 |
| NAV reference (date/share class unverified) | n/a |
| Category | n/a |
| Fund family | n/a |

These optional fields are descriptive and do not change the dislocation score. Missing values are shown as n/a. Metadata may differ in date or units from the price history.

## Diagnostics

- Fund metadata coverage: 3/7 optional fields available; missing fields are not estimated.
- Data coverage score reflects price history and basic identifiers, not complete fund valuation data.
- Provider-reported fund P/E is descriptive; holdings methodology and valuation date are not verified.
- No separate NAV supplied; no NAV premium or discount is measured.
- Fund units may provide equity exposure, but are not corporate shares. Corporate DCF is not applicable to the fund itself; holdings-level fair values and weights would be required for look-through valuation.

## Model note

The score is designed to detect potentially attractive dislocations, not to forecast an exact market bottom. Interpret scores alongside input coverage and, for corporate equities, fundamental quality; neither establishes predictive accuracy.

## Research notice

RiskOptima is for research and educational use only. Reports and scores are not investment advice or recommendations to buy, sell or hold any instrument. Models and third-party data may be incomplete or inaccurate. Past performance does not guarantee future results. Independently verify inputs and results before making investment decisions.

Generated by RiskOptima v2.8.1


---

## Single-ticker equity example

For MSFT, equity metadata activates fundamental scoring. The simplified DCF discounts the supplied FCF proxy, without normalized FCFE or net borrowing. Margin of safety is `(model_value - price) / model_value`; upside divides by price instead. Annualized convergence assumes the current model value is reached over the selected horizon, excludes distributions, and is not a return forecast. Missing cash flow or shares results in unavailable DCF fields, not a fabricated value.


In [11]:
equity_report = analyze_symbol("MSFT", config=config, provider=provider)
assert equity_report.instrument_type == "EQUITY"
if equity_report.intrinsic_value is not None:
    assert abs(equity_report.margin_of_safety - (1 - equity_report.price / equity_report.intrinsic_value)) < 1e-10
display(Markdown(generate_markdown_report(equity_report)))
payload = equity_report.as_dict()
assert "riskoptima_version" in payload


# RiskOptima Dislocation Report — Microsoft Corporation

**Ticker:** `MSFT`  
**As of:** 2026-10-09  
**Instrument type:** EQUITY  
**Valuation mode:** fundamental  
**Price:** 535.07 USD  

## Executive signal

- **Opportunity score:** 38.9/100
- **Classification:** EXTENDED
- **Data coverage:** 100.0/100 (not a probability of accuracy)
- **Dislocation score:** 10.3/100
- **Regime opportunity:** 61.7/100

## Generated interpretation

Microsoft Corporation (MSFT) scores 38.9/100 and is classified as extended. Fundamental quality is 96.3/100 while the valuation score is 20.4/100. The simplified FCF-proxy scenario value is 52.34, 90.2% below the observed price. This is not a reconciled FCFE valuation. The market-dislocation component is 10.3/100 and the regime-opportunity component is 61.7/100. Data coverage is 100.0/100, not a calibrated probability of model accuracy.

## Market dislocation

| Metric | Value |
|---|---:|
| 1Y drawdown | -0.5% |
| Period drawdown | -0.5% |
| 200D trend deviation | 23.7% |
| Price z-score | 1.83 |
| 3M return | 39.3% |
| 12M return | 3.3% |
| Annualized volatility | 39.6% |
| Volatility percentile | 93.2% |
| Regime | neutral |

## Fundamental valuation

| Metric | Value |
|---|---:|
| Quality score | 96.3/100 |
| Valuation score | 20.4/100 |
| FCF yield | 0.4% |
| Earnings yield | 3.4% |
| Forward P/E | 22.60 |
| EV / EBITDA | 20.25 |
| Intrinsic value | 52.34 |
| Margin of safety | -922.3% |
| Upside to model value | -90.2% |
| Implied 5Y annualized return to model value | -37.2% |
| DCF growth assumption | 12.0% |
| DCF discount rate | 9.0% |
| Terminal growth | 2.5% |

> The intrinsic-value estimate is a scenario model, not a precise fair-price claim. It discounts an unadjusted FCF proxy, not reconciled FCFE; net borrowing is not modeled. Margin of safety is (model value - price) / model value. Upside uses price as the denominator. The annualized figure assumes convergence to today's model value over the stated horizon, excludes distributions, and is not an expected-return forecast. Sensitivity to cash flow, growth and discount rates is material.

## Diagnostics

- DCF uses an unadjusted free-cash-flow proxy; net borrowing and normalized FCFE are not modeled.

## Model note

The score is designed to detect potentially attractive dislocations, not to forecast an exact market bottom. Interpret scores alongside input coverage and, for corporate equities, fundamental quality; neither establishes predictive accuracy.

## Research notice

RiskOptima is for research and educational use only. Reports and scores are not investment advice or recommendations to buy, sell or hold any instrument. Models and third-party data may be incomplete or inaccurate. Past performance does not guarantee future results. Independently verify inputs and results before making investment decisions.

Generated by RiskOptima v2.8.1


## Optional LLM narrative adapter

Applications can inject any callable accepting the report context and returning text. This keeps the quantitative model deterministic while allowing the final narrative to be genuinely generative.

```python
def my_generator(context: dict) -> str:
    # Call your preferred model/service here.
    return "..."

report = analyze_symbol("MSFT", narrative_generator=my_generator)
```
